In [1]:
from pathlib import Path
from PIL import Image
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

DATA_DIR = Path(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data")
SRC = DATA_DIR / "split_real_fake"
DST = DATA_DIR / "split_real_fake_norm"
SIZE, QUALITY = 256, 95

kept, dropped = {}, {}
for p in SRC.rglob("*.jpg"):
    split, label = p.parts[-3], p.parts[-2]
    key = (split, label)
    with Image.open(p) as im:
        im = im.convert("RGB")
        w, h = im.size
        if min(w, h) < SIZE:                       # too small: skip it
            dropped[key] = dropped.get(key, 0) + 1
            continue
        s = min(w, h)                              # centre-crop to a square
        left, top = (w - s) // 2, (h - s) // 2
        im = im.crop((left, top, left + s, top + s))
        if s != SIZE:
            im = im.resize((SIZE, SIZE), Image.LANCZOS)
        out = DST / split / label
        out.mkdir(parents=True, exist_ok=True)
        im.save(out / p.name, "JPEG", quality=QUALITY)
    kept[key] = kept.get(key, 0) + 1

print("kept   :", dict(sorted(kept.items())))
print("dropped:", dict(sorted(dropped.items())))

kept   : {('test', 'fake'): 373, ('test', 'real'): 367, ('train', 'fake'): 1749, ('train', 'real'): 1684, ('val', 'fake'): 373, ('val', 'real'): 361}
dropped: {('test', 'real'): 24, ('train', 'real'): 128, ('val', 'real'): 29}


In [2]:
DATA_DIR = Path(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data")

rows = []
for p in (DATA_DIR / "split_real_fake_norm").rglob("*.jpg"):
    with Image.open(p) as im:
        w, h = im.size
    rows.append({"label": p.parts[-2], "w": w, "h": h, "kb": p.stat().st_size / 1024})
meta = pd.DataFrame(rows)

print(meta.groupby("label")[["w", "h"]].agg(["min", "max"]), "\n")
print("Accuracy from width, height and file size:",
      round(cross_val_score(DecisionTreeClassifier(max_depth=3), meta[["w", "h", "kb"]], meta["label"], cv=5).mean(), 3))
print("Accuracy from file size only             :",
      round(cross_val_score(DecisionTreeClassifier(max_depth=3), meta[["kb"]], meta["label"], cv=5).mean(), 3))

         w         h     
       min  max  min  max
label                    
fake   256  256  256  256
real   256  256  256  256 

Accuracy from width, height and file size: 0.977
Accuracy from file size only             : 0.977


In [3]:
DATA_DIR = Path(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data")

final = pd.read_csv(DATA_DIR / "final_image_dataset.csv")
print(pd.crosstab(final["label"], final["gender"]), "\n")
print(pd.crosstab(final["label"], final["age_group"]))

gender  female  male  unknown
label                        
fake         0     0     2495
real      1453  1023      117 

age_group  18-25  26-35  36-50  50+  not_applicable  under_18
label                                                        
fake           0      0      0    0            2495         0
real         330   1484    338  219             143        79


In [1]:
from pathlib import Path
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_DIR = Path(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data")
NORM_DIR = DATA_DIR / "split_real_fake_norm"

final = pd.read_csv(DATA_DIR / "final_image_dataset.csv", dtype=str)

# all normalized REAL images (from any split) with their path
rows = [{"image_id": p.stem, "path": p} for p in NORM_DIR.rglob("real/*.jpg")]
real = pd.DataFrame(rows).merge(final, on="image_id", how="left")
print("normalized real images:", len(real), "| missing labels:", real["gender"].isna().sum(), "\n")

tasks = {
    "split_gender_real": ("gender",    ["female", "male"]),
    "split_age_real":    ("age_group", ["18-25", "26-35", "36-50", "50+", "under_18"]),
}

for folder, (column, classes) in tasks.items():
    df = real[real[column].isin(classes)]
    train_df, rest_df = train_test_split(df, test_size=0.30, stratify=df[column], random_state=42)
    val_df, test_df = train_test_split(rest_df, test_size=0.50, stratify=rest_df[column], random_state=42)

    root = DATA_DIR / folder
    shutil.rmtree(root, ignore_errors=True)        # deletes only this NEW folder
    for split, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
        for _, r in part.iterrows():
            out = root / split / r[column]
            out.mkdir(parents=True, exist_ok=True)
            shutil.copy(r["path"], out / (r["image_id"] + ".jpg"))

    summary = pd.concat([train_df.assign(split="train"), val_df.assign(split="val"), test_df.assign(split="test")])
    print(folder)
    print(pd.crosstab(summary[column], summary["split"]), "\n")

normalized real images: 2412 | missing labels: 0 

split_gender_real
split   test  train  val
gender                  
female   205    953  204
male     140    653  140 

split_age_real
split      test  train  val
age_group                  
18-25        49    227   48
26-35       206    962  206
36-50        43    200   43
50+          32    146   31
under_18     11     53   12 



In [ ]:
import random
import shutil
from pathlib import Path

DATA_DIR = Path(
    r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data"
)

FOLDERS = [
    "split_real_fake",
    "split_gender",
    "split_age"
]

random.seed(42)


def count(root):
    return {
        s: {
            c.name: len(list(c.glob("*.jpg")))
            for c in sorted((root / s).iterdir())
            if c.is_dir()
        }
        for s in ["train", "val", "test"]
    }


for name in FOLDERS:
    root = DATA_DIR / name
    marker = root / "SPLIT_70_20_10.txt"

    if not root.exists():
        print(name, ": folder not found, skipped")
        continue

    if marker.exists():
        print(name, ": already converted, skipped")
        continue

    print(name, "before:", count(root))

    for class_dir in sorted((root / "val").iterdir()):
        if not class_dir.is_dir():
            continue

        files = sorted(class_dir.glob("*.jpg"))

        # Move one-third of validation images to test
        selected_files = random.sample(
            files,
            round(len(files) / 3)
        )

        test_class_dir = root / "test" / class_dir.name
        test_class_dir.mkdir(parents=True, exist_ok=True)

        for f in selected_files:
            shutil.move(
                str(f),
                str(test_class_dir / f.name)
            )

    marker.write_text(
        "One third of validation images moved to test."
    )

    print(name, "after:", count(root), "\n")


split_real_fake : already converted, skipped
split_real_fake_norm : already converted, skipped
split_gender before: {'train': {'female': 1001, 'male': 724, 'unknown': 1836}, 'val': {'female': 228, 'male': 152, 'unknown': 383}, 'test': {'female': 224, 'male': 147, 'unknown': 393}}
split_gender after: {'train': {'female': 1001, 'male': 724, 'unknown': 1836}, 'val': {'female': 152, 'male': 101, 'unknown': 255}, 'test': {'female': 300, 'male': 198, 'unknown': 521}} 

split_age : already converted, skipped
